# Notebook 8: Custom Training Loops
**Estimated time:** 40-50 min
**Prerequisites:** Notebooks 1-7

---
## What you will learn
1. Anatomy of a production-grade training loop
2. Saving and loading model checkpoints
3. Learning rate schedulers — when and how to use them
4. Gradient clipping — preventing exploding gradients
5. Mixed precision training (`torch.cuda.amp`) — 2x speedup on GPU
6. Putting it all together in a clean `Trainer` class

## 1. Checkpoint Saving and Loading

**Checkpointing** = saving the model's state during training so you can:
- Resume training after interruption
- Use the best model (not the last one)
- Deploy a specific version

Always save at minimum:
- `model.state_dict()` — all weights
- `optimizer.state_dict()` — optimizer momentum/state
- Current epoch and best metric

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

torch.manual_seed(42)

# Simple model for demonstration
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- SAVING ---
os.makedirs('./checkpoints', exist_ok=True)

checkpoint = {
    'epoch'     : 5,
    'model_state'    : model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'val_loss'  : 0.234,
    'val_acc'   : 0.912,
}
torch.save(checkpoint, './checkpoints/model_epoch5.pt')
print('Checkpoint saved!')

# --- LOADING ---
ckpt = torch.load('./checkpoints/model_epoch5.pt', map_location='cpu')
model.load_state_dict(ckpt['model_state'])
optimizer.load_state_dict(ckpt['optimizer_state'])
start_epoch = ckpt['epoch'] + 1
best_val    = ckpt['val_loss']
print(f'Resumed from epoch {ckpt["epoch"]}, val_loss={best_val:.3f}')

In [ ]:
# Best-model checkpoint pattern
class BestModelSaver:
    '''Saves the model whenever val_loss improves.'''
    def __init__(self, path, mode='min'):
        self.path = path
        self.mode = mode
        self.best = float('inf') if mode == 'min' else -float('inf')

    def __call__(self, model, optimizer, epoch, metric):
        improved = (self.mode == 'min' and metric < self.best) or                    (self.mode == 'max' and metric > self.best)
        if improved:
            self.best = metric
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'metric': metric,
            }, self.path)
            print(f'  Checkpoint saved (epoch={epoch}, metric={metric:.4f})')
            return True
        return False

saver = BestModelSaver('./checkpoints/best_model.pt', mode='min')
# Usage: saver(model, optimizer, epoch, val_loss)

## 2. Learning Rate Schedulers

The learning rate controls how big each gradient step is:
- Too large → training is unstable, loss explodes
- Too small → training is very slow

**Schedulers** automatically adjust the LR during training.

| Scheduler | When to use |
|-----------|-------------|
| `StepLR` | Decay by factor every N epochs |
| `CosineAnnealingLR` | Smoothly decay from max to min LR |
| `OneCycleLR` | Start low, ramp up, then decay — very fast convergence |
| `ReduceLROnPlateau` | Reduce LR when val_loss stops improving |

In [ ]:
import matplotlib.pyplot as plt

# Visualize different schedulers
model_tmp = nn.Linear(10, 2)

schedulers = {
    'StepLR(step=10, gamma=0.5)': optim.lr_scheduler.StepLR(
        optim.Adam(model_tmp.parameters(), lr=0.1), step_size=10, gamma=0.5),
    'CosineAnnealing(T_max=50)': optim.lr_scheduler.CosineAnnealingLR(
        optim.Adam(model_tmp.parameters(), lr=0.1), T_max=50),
    'ExponentialLR(gamma=0.95)': optim.lr_scheduler.ExponentialLR(
        optim.Adam(model_tmp.parameters(), lr=0.1), gamma=0.95),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, sched) in zip(axes, schedulers.items()):
    lrs = []
    for _ in range(60):
        lrs.append(sched.get_last_lr()[0])
        sched.step()
    ax.plot(lrs)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('LR')
    ax.grid(True, alpha=0.3)
plt.suptitle('Learning Rate Schedules')
plt.tight_layout()
plt.show()

## 3. Gradient Clipping

**Imagine your GPS tells you to turn 180 degrees — you'd just crash.**
Sometimes gradients become huge (exploding gradients). This makes weights jump wildly and training crashes.

**Gradient clipping** caps the gradient norm before the optimizer step:
```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

This doesn't stop learning — it just prevents any single update from being catastrophically large.
Most important for **RNNs** and **Transformers**, but good practice everywhere.

In [ ]:
# Gradient clipping in practice
model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 2))
optimizer = optim.Adam(model.parameters(), lr=1e-3)

x = torch.randn(32, 10)
y = torch.randint(0, 2, (32,))

optimizer.zero_grad()
loss = nn.CrossEntropyLoss()(model(x), y)
loss.backward()

# Check gradient norm BEFORE clipping
total_norm_before = 0
for p in model.parameters():
    if p.grad is not None:
        total_norm_before += p.grad.data.norm(2).item() ** 2
total_norm_before = total_norm_before ** 0.5
print(f'Grad norm BEFORE clip: {total_norm_before:.4f}')

# Clip gradients
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# Check AFTER
total_norm_after = 0
for p in model.parameters():
    if p.grad is not None:
        total_norm_after += p.grad.data.norm(2).item() ** 2
total_norm_after = total_norm_after ** 0.5
print(f'Grad norm AFTER  clip: {total_norm_after:.4f}')

optimizer.step()

## 4. Mixed Precision Training

Modern GPUs have special hardware for **float16** (half precision) that runs 2-4x faster than float32.

**The challenge:** float16 has less precision — some operations overflow or underflow.

**AMP (Automatic Mixed Precision)** solves this: PyTorch automatically decides which operations use float16 vs float32. A **GradScaler** prevents underflow by scaling the loss before backward.

Works out of the box on modern NVIDIA GPUs (and Apple Silicon via MPS).

In [ ]:
from torch.cuda.amp import autocast, GradScaler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# AMP is most effective on CUDA; on CPU it's a no-op but the code still works
model = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 10)).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scaler    = GradScaler(enabled=(DEVICE.type == 'cuda'))  # only active on GPU

for step in range(3):
    x = torch.randn(128, 64).to(DEVICE)
    y = torch.randint(0, 10, (128,)).to(DEVICE)

    optimizer.zero_grad()

    # autocast: use float16 where safe
    with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
        logits = model(x)
        loss   = nn.CrossEntropyLoss()(logits, y)

    # scaler: scale loss to prevent float16 underflow
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)                       # unscale before clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    print(f'Step {step+1}: loss={loss.item():.4f}')

## 5. A Production-Grade Trainer

Let's package everything into a clean `Trainer` class that you can reuse across projects.

In [ ]:
import time

class Trainer:
    def __init__(self, model, optimizer, criterion, scheduler=None,
                 device='cpu', checkpoint_path='./checkpoints/best.pt',
                 grad_clip=1.0, use_amp=False):
        self.model     = model.to(device)
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler
        self.device    = device
        self.grad_clip = grad_clip
        self.scaler    = GradScaler(enabled=use_amp and device == 'cuda')
        self.use_amp   = use_amp
        self.saver     = BestModelSaver(checkpoint_path, mode='min')
        self.history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    def _run_epoch(self, loader, training):
        self.model.train(training)
        total_loss, correct, total = 0, 0, 0

        for X, y in loader:
            X, y = X.to(self.device), y.to(self.device)

            if training:
                self.optimizer.zero_grad()

            with autocast(device_type=self.device if isinstance(self.device, str) else self.device.type,
                          enabled=self.use_amp):
                logits = self.model(X)
                loss   = self.criterion(logits, y)

            if training:
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()

            total_loss += loss.item() * len(X)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += len(X)

        return total_loss / total, correct / total

    def fit(self, train_loader, val_loader, epochs):
        for epoch in range(1, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc = self._run_epoch(train_loader, training=True)
            vl_loss, vl_acc = self._run_epoch(val_loader,   training=False)

            if self.scheduler:
                self.scheduler.step()

            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vl_loss)
            self.history['train_acc'].append(tr_acc)
            self.history['val_acc'].append(vl_acc)

            self.saver(self.model, self.optimizer, epoch, vl_loss)
            elapsed = time.time() - t0
            print(f'Epoch {epoch:3d} [{elapsed:.1f}s] | '
                  f'train {tr_loss:.4f}/{tr_acc:.4f} | val {vl_loss:.4f}/{vl_acc:.4f}')

    def plot(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(self.history['train_loss'], label='Train')
        ax1.plot(self.history['val_loss'],   label='Val')
        ax1.set_title('Loss')
        ax1.legend()
        ax2.plot(self.history['train_acc'], label='Train')
        ax2.plot(self.history['val_acc'],   label='Val')
        ax2.set_title('Accuracy')
        ax2.legend()
        plt.tight_layout()
        plt.show()

print('Trainer class defined.')

In [ ]:
# Use the Trainer on MNIST
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, random_split

transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
full_train = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_set   = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_set, val_set = random_split(full_train, [50000, 10000])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)

# Same model from Notebook 7
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(64, 10)
        )
    def forward(self, x): return self.net(x)

model     = MNISTModel()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, criterion, scheduler,
                  device=DEVICE, checkpoint_path='./checkpoints/mnist_best.pt')
trainer.fit(train_loader, val_loader, epochs=10)
trainer.plot()

---
## YOUR TURN — Mini Project

**Task:** Use the `Trainer` class to train a model and experiment with schedulers.

**Steps:**
1. Load the Fashion-MNIST dataset from Notebook 7
2. Build a CNN classifier (same architecture as Notebook 7)
3. Train the same model **three times** with different schedulers:
   - No scheduler (constant LR = 0.001)
   - `CosineAnnealingLR(T_max=15)`
   - `OneCycleLR(max_lr=0.01, steps_per_epoch=len(train_loader), epochs=15)`
4. Plot all three validation loss curves on one graph
5. Which converges fastest? Which achieves the best final accuracy?

**Hint for OneCycleLR:** it steps after each batch, not each epoch. Set `scheduler.step()` inside the training batch loop, not at epoch end. You'll need to modify `Trainer._run_epoch` slightly for this one.

In [ ]:
# YOUR CODE HERE
# Load FashionMNIST

# Define model, criterion

# Experiment 1: No scheduler

# Experiment 2: CosineAnnealingLR

# Experiment 3: OneCycleLR

# Compare on one plot